# PowerIt (Java)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/PowerIt_java.ipynb)

Chapter 18 — Recursive versus iterative algorithms. Runnable notebook of `PowerIt`: the algorithm in Java, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/18-recursive-vs-iterative/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/PowerIt_python.ipynb) and [C++](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/PowerIt_cpp.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative_java.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. The first cell checks that the JDK is there and installs it if it is missing. Each Java program is written to a file by a `%%writefile` cell, then compiled with `javac` and run with `java` in the next cell.

**Indices.** In the pseudo-code of the notes arrays are indexed from 1; in the code (as in all the languages used here) from 0: the element `a[i]` of the pseudo-code is the element of index `i-1` in the code, and a range `a[l..r]` becomes `a[l-1..r-1]`. The printed output of the notebooks uses indices from 0.

**Names.** The functions have the name of the pseudo-code, in the convention of the language (as in the introductory chapters): `FibRec` → `fibRec` (camelCase).

## Setup

On Colab the JDK is usually already there: the cell below checks it and installs it if it is missing.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## Fast power: recursive and PowerIt (square-and-multiply)

- **FastPower**$(x,n)$ (capitolo 14) — recursive: $x^n=(x^{\lfloor n/2\rfloor})^2$ if $n$ is even, $(x^{\lfloor n/2\rfloor})^2\cdot x$ if it is odd. $\Theta(\log_2 n)$ steps, $\lfloor\log_2 n\rfloor+2$ calls (depth $\Theta(\log_2 n)$). The result of the recursive call is stored in $y$ and used twice.
- **PowerIt**$(x,n)$ — *Input*: a number $x$ and a natural number $n$. *Output*: $x^n$. It reads the bits of $n$ from the least significant one: $\lfloor\log_2 n\rfloor+1$ iterations, extra memory $\Theta(1)$; $p\cdot y^k=x^n$ holds at every iteration.

With $x=2$ and $n=13=(1101)_2$ we print the trace of the chapter: $p$ is multiplied by $x$, $x^4$ and $x^8$ (the bits of 13 equal to 1) and $p=x^{13}$ after $4$ iterations. Then the table: same result, $\lfloor\log_2 n\rfloor+1$ iterations against $\lfloor\log_2 n\rfloor+2$ calls, against the $n$ multiplications of the naive loop.

In [2]:
%%writefile Power.java
import java.util.*;

public class Power {
    static int powCalls = 0;   // calls of FastPower
    static int powIters = 0;   // iterations of PowerIt

    // fastPower(x, n): x^n by recursion on the halving of n
    static long fastPower(long x, int n) {
        powCalls++;
        if (n == 0) {
            return 1;                   // base case
        }
        long y = fastPower(x, n / 2);   // one recursive call
        if (n % 2 == 0) {
            return y * y;               // n even
        }
        return x * y * y;               // n odd
    }

    // powerIt(x, n): x^n with square-and-multiply
    static long powerIt(long x, int n) {
        long p = 1;
        long y = x;
        int k = n;
        while (k > 0) {
            powIters++;
            if (k % 2 == 1) {
                p = p * y;
            }
            y = y * y;
            k = k / 2;
        }
        return p;
    }

    static void powerItTrace(long x, int n) {
        System.out.println("PowerIt(" + x + ", " + n + "): " + n + " = (" + Integer.toBinaryString(n) + ")_2");
        long p = 1, y = x;
        int k = n, it = 0;
        while (k > 0) {
            it++;
            String par = k % 2 == 1 ? "odd" : "even";
            if (k % 2 == 1) {
                p = p * y;
            }
            y = y * y;
            System.out.println(it + ". k=" + k + " " + par + ": p=" + p + ", y=" + y + ", k=" + (k / 2));
            k = k / 2;
        }
        System.out.println("iterations = " + it + ", result p = " + p);
    }

    public static void main(String[] args) {
        powerItTrace(2, 13);
        System.out.println();
        System.out.printf("%3s %12s %8s %10s %6s   (iterations, calls, multiplications)%n", "n", "2^n", "PowerIt", "FastPower", "naive");
        int[] ns = {0, 1, 2, 5, 13, 30};
        for (int n : ns) {
            powIters = 0;
            long r1 = powerIt(2, n);
            powCalls = 0;
            long r2 = fastPower(2, n);
            System.out.printf("%3d %12d %8d %10d %6d   %s%n", n, r1, powIters, powCalls, n, (r1 == r2 && r1 == (1L << n)) ? "OK" : "ERROR");
        }
    }
}

Writing Power.java


In [3]:
!javac Power.java && java Power

PowerIt(2, 13): 13 = (1101)_2
1. k=13 odd: p=2, y=4, k=6
2. k=6 even: p=2, y=16, k=3
3. k=3 odd: p=32, y=256, k=1
4. k=1 odd: p=8192, y=65536, k=0
iterations = 4, result p = 8192

  n          2^n  PowerIt  FastPower  naive   (iterations, calls, multiplications)
  0            1        0          1      0   OK
  1            2        1          2      1   OK
  2            4        2          3      2   OK
  5           32        3          4      5   OK
 13         8192        4          5     13   OK
 30   1073741824        5          6     30   OK


**Try it yourself.** Try another base and another exponent (in C++ and Java the result must fit in 64 bits).

In [4]:
%%writefile ProvaPower.java
import java.util.*;

public class ProvaPower {
    public static void main(String[] args) {
        // Try another base and another exponent (beware: in C++ and Java the result must fit in 64 bits).
        Power.powIters = 0;
        Power.powCalls = 0;
        long r1 = Power.powerIt(3, 25);
        long r2 = Power.fastPower(3, 25);
        System.out.println(r1 + " " + Power.powIters + " " + r2 + " " + Power.powCalls);
    }
}

Writing ProvaPower.java


In [5]:
!javac ProvaPower.java && java ProvaPower

847288609443 5 847288609443 6


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).